## 深度优先搜索与强连通分量

## 环境配置说明：
### 如果在本地运行：
#### 首次配置
* 安装Anaconda：访问 https://www.anaconda.com/products/distribution
* 创建conda环境：`conda create -n cs131232 python=3.8`
  * 详见 https://conda.io/projects/conda/en/latest/user-guide/tasks/manage-environments.html
* 激活环境：`conda activate cs131232`
* 安装pip：`conda install pip`
* 安装Jupyter：`conda install jupyter`
#### 每次启动
* 确保激活环境：`conda activate cs131232`
* 启动Jupyter：`jupyter notebook` 或 `jupyter lab`
* 运行以下单元格**并在需要时重启内核**

In [ ]:
# Install dependencies
!pip install -r requirements.txt --quiet

In [ ]:
import otter
assert (
    otter.__version__ >= "4.4.1"
), "Please reinstall the requirements and restart your kernel."
import networkx as nx
import typing
import numpy as np
import tqdm
import pickle
grader = otter.Notebook("dfs-scc.ipynb")

rng_seed = 0

In [ ]:
# Load test cases
file_path = "generated_testcases.pkl"

# Load the variables from the pickle file
with open(file_path, "rb") as file:
    loaded_data = pickle.load(file)
file.close()
inputs, outputs = loaded_data

#### 在代码中表示图

在代码中表示图的方式有多种。在课堂上，我们介绍了邻接矩阵和邻接表。此外，还有一种边列表表示法，即将所有边存储在一个一维列表中。在本课程以及大多数情况下，我们通常选择使用邻接表表示法，因为它能够高效地查找一个节点的邻居。

在许多编程问题中，为了方便起见，顶点通常被标记为从 $0$ 到 $n-1$（回想一下，大多数编程语言中的数组和列表是从索引 $0$ 开始的）。这使得我们可以使用一个存储整数的列表的列表来表示邻接表。给定一个边列表，以下代码将为**无权有向图**创建一个邻接表。

In [ ]:
def generate_adj_list(n, edge_list):
    """
    args:
        n:int = number of nodes in the graph. The nodes are labelled with integers 0 through n-1
        edge_list:List[Tuple(int,int)] = edge list where each tuple (u,v) represents the directed 
            edge (u,v) in the graph
    return:
        A List[List[int]] representing the adjacency list 
    """
    adj_list = [[] for i in range(n)] 
    for u, v in edge_list:
        adj_list[u].append(v)
    for nodes in adj_list:
        nodes.sort()
    return adj_list

def draw_graph(adj_list):
    """Utility method for visualizing graphs

    args:
        adj_list (List[List[int]]): adjacency list of the graph given by generate_adj_list
    """
    G = nx.DiGraph()
    for u in range(len(adj_list)):
        for v in adj_list[u]:
            G.add_edge(u, v)
    nx.draw(G, with_labels=True)

## 问题 1.1: 重构深度优先搜索路径

在课堂上，我们展示了如何使用深度优先搜索（DFS）来检查两个节点之间是否存在路径、对节点进行拓扑排序以及寻找强连通分量。在这些算法中，我们使用了前序编号和后序编号。

在这里，你将实现DFS的一个变体，以打印出两个节点之间的路径。在许多问题中，我们希望找到两个节点之间的具体路径，而不仅仅是判断路径是否存在。

> **任务 1:** 使用DFS计算从 $s$ 到 $t$ 的路径，并将该路径表示为节点列表返回。

例如，路径 ${s \to a \to b \to c \to t}$ 对应于列表 `[s, a, b, c, t]`。 如果不存在路径，则返回空列表 `[]`。

无需为该问题实现前序和后序编号的计算。

*提示:*

1. 如果你想从课堂上讲的递归DFS实现开始，则可以使用[mutable types or the `nonlocal` keyword]来保留在递归调用中的状态。

2. 维护一个额外的数据结构，以跟踪每次访问新节点时的前一个节点，可能会有所帮助。

_分值：_ 20

In [ ]:
def dfs_path(adj_list, s, t):
    """
    args:
        adj_list:List[List] = an adjacency list 
        s:int = an int representing the starting node
        t:int = an int representing the destination node

    return: 
        a list of nodes starting with s and ending with t representing an s to t path if it exists. 
        Returns an empty list otherwise
    """
    n = len(adj_list)
    prev = [-1] * n
    visited = [False] * n
    visited[s] = True
    stack = [s]

    # Iterative DFS; prev records the tree edge used to discover each node.
    while stack:
        u = stack.pop()
        if u == t:
            break
        for v in reversed(adj_list[u]):
            if not visited[v]:
                visited[v] = True
                prev[v] = u
                stack.append(v)

    if not visited[t]:
        return []

    path = []
    node = t
    while node != -1:
        path.append(node)
        node = prev[node]
    path.reverse()
    return path

### 调试

你可以在下面的单元格中创建一些测试用例，以帮助调试你的解决方案。我们提供了一些简单的测试作为示例，但这些测试可能并不全面。

要在测试中添加一个新图，请按照下一个单元格中的示例，将新的边列表追加到 `edge_lists` 中。

__请注意，这些边是有向的，因此请勿将边的两个方向都添加到边列表中。__

In [ ]:
edge_lists = []
edge_lists.append([(0,1), (0,2), (1,2), (2,3), (3,4), (3,5), (4,5)])   # edge list of first graph
edge_lists.append([(0,1), (0,2), (1,2), (3,4), (3,5), (4,5)])          # edge list of second graph
# add any additional tests here

对于每个测试用例，你还需要添加一个起始节点 $s$、一个目标节点 $t$，以及图中的节点总数 $n$，并将它们添加到以下列表中。

In [ ]:
s_list = []
s_list.append(0)  # s for first graph 
s_list.append(1)  # s for second graph 
# add any additional tests here

t_list = []
t_list.append(3)  # t for first graph
t_list.append(4)  # t for second graph
# add any additional tests here

n_list = []
n_list.append(6)  # n = 6 for first graph
n_list.append(6)  # n = 6 for second graph
# add any additional tests here

以下是一个自动评分器的简化版本。你可能需要添加更多的打印语句或其他调试信息，以便检查你的函数的运行情况。

In [ ]:
import matplotlib.pyplot as plt
index = 1
for s, t, n, edge_list in zip(s_list, t_list, n_list, edge_lists):
    print("Testing graph:", index)
    index += 1
    
    adj_list_graph = generate_adj_list(n, edge_list) # function defined earlier
    
    path = dfs_path(adj_list_graph, s, t) 
    
    nx_graph = nx.DiGraph(edge_list)
    
    # uncomment the following to plot each graph
    '''
    nx.draw(nx_graph, with_labels=True)
    plt.title(f"Graph with {n} vertices and start node {s} and destination {t}")
    plt.show()
    '''
    
    if not nx.has_path(nx_graph,s,t):
        assert len(path) == 0, f"your dfs_path found an s-t path when there isn't one."
    else:
        # checks that the path returned is a real path in the graph and that it starts and ends 
        # at the right vertices
        assert nx.is_simple_path(nx_graph, path), f"your dfs_path did not return a valid simple path"
        assert path[0] == s, f"your dfs_path returned a valid simple path, but it does not start at node s"
        assert path[-1] == t, f"your dfs_path returned a valid simple path, but it does not end at node t"

print("Success")

In [ ]:
grader.check("q1.1")

## 问题 1.2: 前序和后序编号

为了进行拓扑排序或寻找强连通分量，我们需要为每个节点计算其前序（pre）和后序（post）编号。

在本题中，你需要修改之前的 DFS 实现，以便为每个节点生成前序和后序编号。建议直接复制前一题的代码，并在此处进行修改。

> **任务 2:** 实现一个函数，计算图中每个节点的 DFS 前序和后序编号。

为了通过自动评分器，最小前序编号应为 1. 最大后序编号应为 $2 \times \text{(顶点数量)}$。请返回两个元组列表：`pre` 列表应包含元组 `(节点, 前序编号)`，而 `post` 列表应包含元组 `(节点, 后序编号)`。

两个列表应按照元组中的前/后序编号自然顺序排列。**禁止使用任何排序函数来实现此要求！**

> **思考:** 为什么以这种方式返回前/后序编号有助于寻找强连通分量？

可以删除初始代码并自行实现所需功能。

**注意:** 本部分不再假设整个图必然可以从某个起点到达。这将如何影响你的实现？

最后，**通过选择数值最小的节点来打破平局**。如果实现逻辑正确但打破平局的方式不同，自动评分器可能会判定为失败。

_分值：_ 15

In [ ]:
def get_pre_post(adj_list):
    """
    args:
        adj_list:List[List[int]] = the adjacency list that represents our input graph
    return:
        List[Tuple(int, int)], List[Tuple(int, int)] representing the pre and post order values
            respectively. Each tuple should have a vertex as its first entry, and the pre/post order
            value as its second entry.
    """
    time = 1
    pre = []
    post = []

    n = len(adj_list)
    visited = [False] * n

    # A global clock shared by all DFS trees. Roots and neighbours are visited
    # in increasing node order, which is the required tie-breaking rule.
    for root in range(n):
        if visited[root]:
            continue
        visited[root] = True
        pre.append((root, time))
        time += 1
        stack = [(root, 0)]

        while stack:
            u, next_index = stack[-1]
            if next_index < len(adj_list[u]):
                v = adj_list[u][next_index]
                stack[-1] = (u, next_index + 1)
                if not visited[v]:
                    visited[v] = True
                    pre.append((v, time))
                    time += 1
                    stack.append((v, 0))
            else:
                stack.pop()
                post.append((u, time))
                time += 1
            
    return pre, post

In [ ]:
grader.check("q1.2")

### 问题2: 强连通分量（Kosaraju-Sharir 算法）

我们现在将看到如何将 DFS 遍历和前后序值的概念结合起来，以获得图中的强连通分量。SCC 元图非常有用，因为它们允许我们构造 DAG、线性化包含环的图，以及获得图中的等价类等。

S. Rao Kosaraju 和 Micha Sharir 独立解决了在图中查找 SCC 的问题，提出了 Kosaraju-Sharir 算法。

该算法的高层概述如下：
1. 在反向图上运行 DFS，并存储每个节点的后序编号。
2. 在原图上运行 DFS，按后序编号递减的顺序访问节点。此次遍历中的每个 DFS 树都是一个 SCC。

我们首先需要的是一个反向图 $G^R$。在这一部分中，我们将实现一个函数，用于反转以邻接表表示的图。

> **任务：** 给定一个以邻接表表示的图，返回表示反向图的邻接表（顶点集与输入图相同，当且仅当输入图的边集中存在 $(u, v)$ 时，反向图的边集中存在 $(v, u)$）。

_分值：_ 15

In [ ]:
def reverse_graph(adj_list):
    """
    args:
        adj_list:List[List[int]] = the adjacency list that represents our input graph
    return:
        List[List[int]] representing the adjacency list of the reversed input graph
    """
    reversed_adj_list = [[] for _ in adj_list]
    for u, neighbors in enumerate(adj_list):
        for v in neighbors:
            reversed_adj_list[v].append(u)

    for neighbors in reversed_adj_list:
        neighbors.sort()
    return reversed_adj_list

In [ ]:
grader.check("q2.1")

现在，你将完成算法的其余部分，以找出图中的强连通分量。

> **任务：**
> * 给定一个以邻接表表示的图，返回一个 SCC 分量列表。按找到 SCC 的顺序返回它们。
> * SCC 分量应以 Python `Set` 表示，且仅包含该 SCC 中存在的节点。

In [ ]:
def find_SCCs(adj_list):
    """
    args:
        adj_list:List[List[int]] = the adjacency list that represents our input graph
    return:
        List(Set(int, ...), ...) a list of sets where each set contains all the nodes 
            that belong to the corresponding SCC
    """
    scc_list = []

    # Kosaraju-Sharir: decreasing post order on the reversed graph, then each
    # DFS tree in the original graph is one SCC.
    reversed_adj_list = reverse_graph(adj_list)
    _, post = get_pre_post(reversed_adj_list)

    n = len(adj_list)
    visited = [False] * n
    for node, _ in reversed(post):
        if visited[node]:
            continue

        component = set()
        visited[node] = True
        stack = [node]
        while stack:
            u = stack.pop()
            component.add(u)
            for v in adj_list[u]:
                if not visited[v]:
                    visited[v] = True
                    stack.append(v)
        scc_list.append(component)

    return scc_list

In [ ]:
grader.check("q2.2")

## 提交

确保在运行下面的单元格之前，你已经按顺序运行了笔记本中的所有单元格，以便所有图像/图表都能在输出中显示。下面的单元格将为你生成一个用于提交的压缩文件。

In [ ]:
grader.export(pdf=False, force_save=True, run_tests=True)